# Chapter 12: Functions and Modules for VLSI Automation

Companion notebook to **python-from-zero** · `lesson-12` · based on *Python for VLSI*, Chapter 12.

### You will learn

- Define functions with positional, default, and keyword args.
- Use `*args` and `**kwargs` for flexible signatures.
- Constrain with `/` (positional-only) and `*` (keyword-only).
- Split code into importable modules and a local package.

In [1]:
import sys, platform
print(f"Python {sys.version.split()[0]} on {platform.system()}")
assert sys.version_info >= (3, 9), "This notebook needs Python 3.9 or newer."

Python 3.12.3 on Linux


## def and return

`def name(params):` introduces a function, `return` sends a value back.

In [2]:
def check_slack(slack: float) -> str:
    if slack < 0:  return "violation"
    if slack == 0: return "critical"
    return "safe"

for s in (-0.05, 0.0, 0.12):
    print(f"{s:+.2f} -> {check_slack(s)}")


-0.05 -> violation
+0.00 -> critical
+0.12 -> safe


## Default arguments

Defaults apply when caller omits them. Never use mutable defaults!

In [3]:
def log(net, severity="low"):
    print(f"{net}: {severity}")

log("clk")
log("rst", "high")

def append_net(name, container=None):
    container = [] if container is None else container
    container.append(name)
    return container

print(append_net("clk"))
print(append_net("rst"))


clk: low
rst: high
['clk']
['rst']


## Positional vs keyword calls

Positional matches by order; keyword by name. Keyword must follow positional.

In [4]:
def report(cell, severity): print(f"{cell}: {severity}")
report("U1", "high")
report(severity="medium", cell="U2")


U1: high
U2: medium


## Positional-only and keyword-only

`/` forces everything before it to be positional-only; `*` forces everything after it to be keyword-only.

In [5]:
def extract(path, /, *, threshold=0.1, unit="ns"):
    print(f"{path} @ {threshold}{unit}")

extract("CLK->FF", threshold=0.05)
try:
    extract(path="CLK->FF")
except TypeError as e:
    print("error:", e)


CLK->FF @ 0.05ns
error: extract() got some positional-only arguments passed as keyword arguments: 'path'


## *args and **kwargs

`*args` collects extra positional args as a tuple; `**kwargs` collects extra keyword args as a dict.

In [6]:
def report_cells(*cells):
    for c in cells: print("checking", c)

report_cells("U1", "U2", "U3")

def configure(**settings):
    for k, v in settings.items():
        print(f"{k:10s} = {v}")

configure(clock="clk", threshold=0.2, mode="hold")


checking U1
checking U2
checking U3
clock      = clk
threshold  = 0.2
mode       = hold


## Write and import a local module

A module is just a `.py` file. Save it, then `import` it.

In [7]:
from pathlib import Path
import importlib, sys

Path("vlsi_utils.py").write_text('''"""Tiny VLSI helpers."""
def check_slack(slack):
    if slack < 0: return "violation"
    if slack == 0: return "critical"
    return "safe"

def parse_violation(line):
    parts = [p.strip() for p in line.split(",")]
    return {"node": parts[0], "drop": float(parts[-1])}
''')

if "" not in sys.path:
    sys.path.insert(0, "")
import vlsi_utils
importlib.reload(vlsi_utils)

print(vlsi_utils.check_slack(-0.02))
print(vlsi_utils.parse_violation("n0, vdd, 0.91, 0.045"))


violation
{'node': 'n0', 'drop': 0.045}


## Build a local package

A package is a directory with `__init__.py`.

In [8]:
from pathlib import Path
import importlib, shutil, sys

pkg = Path("mytools")
if pkg.exists(): shutil.rmtree(pkg)
pkg.mkdir()
(pkg / "__init__.py").write_text("")
(pkg / "math_utils.py").write_text("def square(n): return n*n\n")
(pkg / "report_utils.py").write_text("def banner(m): print(f'[REPORT] {m}')\n")

if "" not in sys.path: sys.path.insert(0, "")
for n in ("mytools","mytools.math_utils","mytools.report_utils"):
    sys.modules.pop(n, None)

from mytools import math_utils, report_utils
print(math_utils.square(7))
report_utils.banner("package imported OK")


49
[REPORT] package imported OK


## VLSI practice — reusable slack reporter

Combine default args, keyword-only options, and `**kwargs` metadata.

In [9]:
def slack_report(paths, *, budget=0.0, top=3, **meta):
    violators = sorted(((n,s) for n,s in paths if s < budget),
                       key=lambda r: r[1])[:top]
    print("== Slack report ==")
    for k,v in meta.items():
        print(f"{k:10s}: {v}")
    if not violators:
        print("no violations under", budget); return
    for rank, (name, slack) in enumerate(violators, 1):
        print(f"{rank}. {name:15s} slack={slack:+.3f} ns")

slack_report(
    [("reg2reg_a",-0.12),("in2reg_b",0.02),("reg2reg_c",-0.30),
     ("reg2out_d",-0.05),("in2reg_e",0.10)],
    budget=0.0, top=3,
    design="cpu_top", tool="primetime", corner="ff_0p88v_125c",
)


== Slack report ==
design    : cpu_top
tool      : primetime
corner    : ff_0p88v_125c
1. reg2reg_c       slack=-0.300 ns
2. reg2reg_a       slack=-0.120 ns
3. reg2out_d       slack=-0.050 ns


### Exercise 1
Write `total(*nums)` returning their sum; test with `1,2,3`.

In [10]:
# Your code here


<details><summary>Show solution</summary>

```python
def total(*n): return sum(n)
print(total(1,2,3))
```

</details>

### Exercise 2
Write `run(stage, *, clock='clk', mode='setup')` and call it two ways.

In [11]:
# Your code here


<details><summary>Show solution</summary>

```python
def run(stage,*,clock='clk',mode='setup'): print(stage,clock,mode)
run('synth'); run('route', clock='clk2', mode='hold')
```

</details>

### Exercise 3
Write `chain(*funcs)(x)` that applies each function in order.

In [12]:
# Your code here


<details><summary>Show solution</summary>

```python
def chain(*fs):
    def run(x):
        for f in fs: x=f(x)
        return x
    return run
print(chain(str.strip, str.upper)('  hi '))
```

</details>

## Recap

- `def`, `return`, type hints; defaults via `param=value`.
- `*args` packs positionals, `**kwargs` packs keywords.
- `/` and `*` in signatures constrain how callers pass args.
- Modules are `.py` files; packages are dirs with `__init__.py`.

## What's next

Continue with **Chapter 13: File I/O Operations for VLSI Automation** in this repo, and the matching lesson on [python-from-zero](https://python-from-zero.pages.dev).